# Day 4: Social Science Applications

**LLMs for Social Science** | Oxford Spring School

## The Course at a Glance

| Day | Theme | What You Learn |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | How models represent meaning, process text, and generate language |
| Day 2 ✓ | From Models to Tools | Post-training (RLHF, DPO), prompting as experimental design, model evaluation |
| Day 3 ✓ | Deploying for Research | Validation, fine-tuning (BERT and LoRA), APIs and working at scale |
| **→ Day 4** | **Social Science Applications** | **Information extraction, RAG, LLMs as simulated agents** |
| Day 5 | Agentic Workflows | Tool use, autonomous research assistants, capstone project |

Each day builds on the previous one. Over the first three days you learned how models work, how to prompt and validate them, and how to fine-tune when prompting is not enough. Today you **apply everything to real social science research tasks** beyond classification.

This matters because:

- **Information extraction** turns unstructured text into structured datasets: the raw material for quantitative analysis.
- **RAG** lets you work with corpora that exceed any model's context window, from parliamentary archives to decades of newspaper coverage.
- **Simulated agents** raise fundamental questions about measurement validity that connect directly to your methodological training.

## Today's outline

- **Section 1:** Information Extraction and Summarization (~30 min)
- *Break (~15 min)*
- **Section 2:** Retrieval-Augmented Generation (RAG) (~70 min)
- **Section 3:** LLMs as Simulated Agents (~25 min)
- **Closing** (~10 min)


## Setup

In [ ]:
#@title Install dependencies
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers faiss-cpu
!pip install -q torch pandas numpy scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
#@title Load the language model: Qwen2.5-7B-Instruct (4-bit)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded: {model_name} (4-bit)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
#@title Generation helper

def generate_response(user_message, max_new_tokens=300, temperature=0.0):
    """Generate a response from the instruct model."""
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
        )

    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [ ]:
#@title UK manifesto corpus

# Excerpts from UK party manifestos (Labour, Conservative, Liberal Democrats)
# across the 2017, 2019, and 2024 general elections, covering key policy areas.
# Each excerpt is a self-contained paragraph suitable for retrieval.

MANIFESTO_CORPUS = [
    # --- ECONOMY ---
    {
        "id": "lab_2017_econ_1",
        "party": "Labour",
        "year": 2017,
        "topic": "economy",
        "text": "Labour will invest in infrastructure and skills to build an economy that works for everyone, not just those at the top. We will establish a National Investment Bank capitalized with 250 billion pounds to channel investment to every region and nation of the UK, supporting one million new jobs."
    },
    {
        "id": "con_2017_econ_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "economy",
        "text": "We will continue to reduce the deficit so that we are no longer spending more than we earn. We will keep taxes as low as possible, maintain our commitment to the triple lock on pensions, and ensure that Britain remains one of the most competitive economies in the world."
    },
    {
        "id": "lib_2017_econ_1",
        "party": "Liberal Democrats",
        "year": 2017,
        "topic": "economy",
        "text": "We will provide a stable, sustainable economic framework that promotes investment and innovation. This means remaining in the single market, reforming business rates to support high street shops, and introducing a one penny rise on income tax to fund the NHS and social care."
    },
    {
        "id": "lab_2019_econ_1",
        "party": "Labour",
        "year": 2019,
        "topic": "economy",
        "text": "Labour will launch the largest investment programme in living memory, investing over 400 billion pounds in infrastructure and industry. This Green Industrial Revolution will create one million climate jobs and bring major industries into public ownership, including rail, mail, water, and energy."
    },
    {
        "id": "con_2019_econ_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "economy",
        "text": "Our economic plan is built on getting Brexit done and unleashing the potential of the whole United Kingdom. We will not raise the rate of income tax, VAT, or National Insurance. We will invest in infrastructure with a 100 billion pound programme and level up every part of the country."
    },
    {
        "id": "lib_2019_econ_1",
        "party": "Liberal Democrats",
        "year": 2019,
        "topic": "economy",
        "text": "Stopping Brexit is the biggest boost we can give the economy. On top of that, we will invest in infrastructure, skills, and research to build a stronger, greener economy. We will introduce a frequent-flyer levy and use the proceeds to fund green transport alternatives."
    },
    {
        "id": "lab_2024_econ_1",
        "party": "Labour",
        "year": 2024,
        "topic": "economy",
        "text": "Labour will deliver economic stability as the foundation for growth. We will maintain strict fiscal rules: the current budget must move into balance and debt must be falling as a share of the economy. We will create a National Wealth Fund to invest alongside the private sector in the industries of the future."
    },
    {
        "id": "con_2024_econ_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "economy",
        "text": "Our plan is to cut taxes for workers and grow the economy. We will abolish the main rate of employee National Insurance entirely by 2030, saving the average worker over 1,300 pounds per year. We will maintain our commitment to fiscal responsibility while reducing the tax burden."
    },

    # --- HEALTHCARE ---
    {
        "id": "lab_2017_health_1",
        "party": "Labour",
        "year": 2017,
        "topic": "healthcare",
        "text": "Labour will guarantee the funding the NHS needs, delivering an extra 30 billion pounds over the next parliament. We will end the crisis in social care with a new National Care Service and ensure that no one has to sell their home to pay for care."
    },
    {
        "id": "con_2019_health_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "healthcare",
        "text": "We will build and fund 40 new hospitals over the next ten years and deliver 50,000 more nurses. We will maintain the triple lock on pensions, introduce an integrated health and social care plan, and ensure that nobody has to sell their home to pay for social care."
    },
    {
        "id": "lab_2024_health_1",
        "party": "Labour",
        "year": 2024,
        "topic": "healthcare",
        "text": "Labour will cut NHS waiting times with 40,000 more evening and weekend appointments each week, paid for by cracking down on tax avoidance and non-dom status. We will recruit thousands more GPs and reform the system to move from hospital to community care, making the NHS fit for the future."
    },
    {
        "id": "lib_2024_health_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "healthcare",
        "text": "We will give everyone the right to see a GP within seven days, or within 24 hours if urgent. We will fix the social care crisis by introducing free personal care, funded by a dedicated Health and Social Care Tax. We will invest in mental health services to ensure parity of esteem with physical health."
    },

    # --- IMMIGRATION ---
    {
        "id": "con_2017_imm_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "immigration",
        "text": "We will reduce immigration to sustainable levels, bringing net migration down to the tens of thousands. We will remain signatories to the European Convention on Human Rights but will consider further reform. Leaving the EU means we can control our borders and decide who comes to Britain."
    },
    {
        "id": "lab_2019_imm_1",
        "party": "Labour",
        "year": 2019,
        "topic": "immigration",
        "text": "Labour will develop a humane immigration policy that recognises the contribution of migrants to our society and economy. We will end indefinite detention, close Yarl's Wood and Brook House detention centres, and ensure that family reunion rights are maintained after Brexit."
    },
    {
        "id": "con_2024_imm_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "immigration",
        "text": "We will cap the overall number of visas issued each year, set by Parliament, so that the British people have democratic control over immigration levels. We will maintain our Rwanda partnership to deter illegal entry and process asylum claims offshore."
    },
    {
        "id": "lab_2024_imm_1",
        "party": "Labour",
        "year": 2024,
        "topic": "immigration",
        "text": "Labour will reduce net migration with a properly managed system. We will reform the points-based immigration system by linking visa issuance to training plans so that employers invest in British workers. We will scrap the Rwanda scheme and instead use the money to fund a new Border Security Command."
    },

    # --- CLIMATE / ENVIRONMENT ---
    {
        "id": "lab_2019_climate_1",
        "party": "Labour",
        "year": 2019,
        "topic": "climate",
        "text": "Labour will launch a Green New Deal to achieve net zero carbon emissions by 2030. We will invest in renewable energy, insulate millions of homes, and create hundreds of thousands of green jobs. Energy and water will be brought into public ownership to deliver clean energy for all."
    },
    {
        "id": "con_2019_climate_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "climate",
        "text": "We will reach net zero greenhouse gas emissions by 2050. We will invest 9.2 billion pounds in the energy efficiency of homes, schools, and hospitals and lead the global fight against climate change. We will establish new national parks and plant 30 million trees."
    },
    {
        "id": "lib_2019_climate_1",
        "party": "Liberal Democrats",
        "year": 2019,
        "topic": "climate",
        "text": "We will reach net zero greenhouse gas emissions by 2045, five years ahead of the current target. We will plant 60 million trees a year, ban fracking, and generate 80 percent of electricity from renewables by 2030. A green future is a Liberal Democrat future."
    },
    {
        "id": "lab_2024_climate_1",
        "party": "Labour",
        "year": 2024,
        "topic": "climate",
        "text": "Labour will set up Great British Energy, a new publicly owned clean energy company headquartered in Scotland. We will double onshore wind, triple solar power, and quadruple offshore wind by 2030. This will cut energy bills, create jobs, and make Britain energy independent."
    },

    # --- EDUCATION ---
    {
        "id": "lab_2017_edu_1",
        "party": "Labour",
        "year": 2017,
        "topic": "education",
        "text": "Labour will abolish university tuition fees and restore maintenance grants for the poorest students. We will introduce a National Education Service for lifelong learning, reduce class sizes for five, six, and seven year olds, and ensure every school has the funding it needs."
    },
    {
        "id": "con_2017_edu_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "education",
        "text": "We will build on our reforms to create more good school places, open new free schools and university technical colleges, and lift the ban on new grammar schools. We will ensure that every child has the opportunity to attend a good or outstanding school, regardless of their background."
    },
    {
        "id": "lib_2024_edu_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "education",
        "text": "We will invest in education from early years through university. We will extend free school meals to all primary school children, recruit more teachers with better pay and conditions, and triple the early years pupil premium to give every child the best start in life."
    },

    # --- HOUSING ---
    {
        "id": "lab_2024_housing_1",
        "party": "Labour",
        "year": 2024,
        "topic": "housing",
        "text": "Labour will build 1.5 million new homes over the next parliament by reforming planning rules, hiring hundreds of new planning officers, and building a new generation of council and social housing. We will introduce a permanent mortgage guarantee scheme and reform the private rental market."
    },
    {
        "id": "con_2024_housing_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "housing",
        "text": "We will deliver 1.6 million new homes by fast-tracking building on brownfield sites and investing in urban regeneration. We will protect the green belt, abolish no-fault evictions, and help young people onto the housing ladder with a new Help to Buy scheme."
    },
    {
        "id": "lib_2024_housing_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "housing",
        "text": "We will build 380,000 new homes a year, including 150,000 social homes, by hiring more planning officers, releasing land held by developers, and establishing a new Housing and Communities Agency. We will end the scandal of homelessness with a comprehensive strategy and proper funding."
    },

    # --- DEFENCE / FOREIGN POLICY ---
    {
        "id": "con_2024_defence_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "defence",
        "text": "We will increase defence spending to 2.5 percent of GDP by 2030, maintain our nuclear deterrent, and continue to lead NATO support for Ukraine. Britain will remain a leading force for global security, with a defence industrial strategy to invest in British manufacturing."
    },
    {
        "id": "lab_2024_defence_1",
        "party": "Labour",
        "year": 2024,
        "topic": "defence",
        "text": "Labour will maintain the UK's nuclear deterrent and commit to NATO. We will set out a path to spending 2.5 percent of GDP on defence and conduct a Strategic Defence Review in the first year of government. We will continue to stand with Ukraine for as long as it takes."
    },
]

# Convert to DataFrame for easy manipulation
corpus_df = pd.DataFrame(MANIFESTO_CORPUS)
print(f"Manifesto corpus: {len(corpus_df)} excerpts")
print(f"Parties: {corpus_df['party'].unique().tolist()}")
print(f"Years: {sorted(corpus_df['year'].unique().tolist())}")
print(f"Topics: {corpus_df['topic'].unique().tolist()}")
print(f"\nSample excerpt:")
print(f"  [{corpus_df.iloc[0]['party']} {corpus_df.iloc[0]['year']}, {corpus_df.iloc[0]['topic']}]")
print(f"  {corpus_df.iloc[0]['text'][:150]}...")


---

# Section 1: Information Extraction and Summarization

In Days 2 and 3, you classified text: the output was one label per document. Many research tasks need more: extracting **structured information** from text. What policy is being proposed? Who benefits? What justification is given? What entities and relationships are mentioned?

This section applies the structured output skills from Day 2 to a harder problem, and introduces a new concern: **faithfulness**. When the model extracts or summarizes, does it stick to what the text actually says, or does it add, omit, or distort?


### Exercise 1a: Structured extraction from manifesto text

Write a prompt that extracts structured information from a manifesto paragraph. The model should return a JSON object with these fields:

- `policy`: What is being proposed (one sentence)
- `position`: The ideological direction (e.g., "increase spending", "cut taxes", "expand public ownership")
- `target_group`: Who benefits or is affected
- `justification`: Why the party says this is needed

Run it on a few excerpts from different parties and topics.


In [ ]:
### Exercise 1a: Structured extraction

# Write an extraction prompt. Use {text} as placeholder.
extraction_prompt = ""  # YOUR CODE HERE

# Test on a few excerpts
if extraction_prompt:
    test_excerpts = corpus_df.sample(5, random_state=42)
    for _, row in test_excerpts.iterrows():
        print(f"[{row['party']} {row['year']}, {row['topic']}]")
        print(f"Text: {row['text'][:100]}...")
        response = generate_response(extraction_prompt.format(text=row['text']), max_new_tokens=200)
        print(f"Extraction: {response[:300]}")
        print()


In [ ]:
#@title Solution 1a

extraction_prompt = (
    "Read the following excerpt from a UK party manifesto and extract structured information.\n\n"
    "Text: {text}\n\n"
    "Respond with ONLY a JSON object in this exact format:\n"
    '{{"policy": "what is being proposed (one sentence)",'
    ' "position": "ideological direction (e.g. increase spending, cut taxes)",'
    ' "target_group": "who benefits or is affected",'
    ' "justification": "why the party says this is needed"}}'
)

test_excerpts = corpus_df.sample(5, random_state=42)
for _, row in test_excerpts.iterrows():
    print(f"[{row['party']} {row['year']}, {row['topic']}]")
    response = generate_response(extraction_prompt.format(text=row['text']), max_new_tokens=200)

    # Try to parse as JSON
    try:
        cleaned = response.strip().strip('`')
        if cleaned.startswith('json'):
            cleaned = cleaned[4:].strip()
        parsed = json.loads(cleaned)
        for k, v in parsed.items():
            print(f"  {k}: {v}")
    except json.JSONDecodeError:
        print(f"  [Raw response]: {response[:200]}")
    print()

print("-> Compare the extractions to the original text. Is the model faithful?")
print("   Does it add information not in the text? Does it miss key details?")
print("   A 7B model will usually get the gist right but may oversimplify or")
print("   inject background knowledge that is not in the excerpt.")


### Exercise 1b: Summarization failure modes

Summarization is extraction's cousin: instead of pulling out specific fields, you condense a longer text into a shorter one. But summarization has distinctive failure modes that matter for research:

- **Hallucination:** the summary includes facts not present in the original
- **Omission:** the summary leaves out important points
- **Distortion:** the summary shifts the emphasis or framing

Your task: read the model's summary alongside the original text and identify which failure modes (if any) are present.


In [ ]:
### Exercise 1b: Summarization and faithfulness evaluation

# Combine multiple excerpts from the same party into a longer text
labour_texts = corpus_df[corpus_df['party'] == 'Labour']['text'].tolist()
combined_text = " ".join(labour_texts[:5])  # first 5 Labour excerpts

print(f"Original text ({len(combined_text.split())} words):")
print(combined_text[:300])
print("...")
print()

# Ask the model to summarize
summary_prompt = (
    "Summarize the following party manifesto excerpts in 3-4 sentences. "
    "Include only information that is explicitly stated in the text.\n\n"
    f"Text: {combined_text}\n\n"
    "Summary:"
)

summary = generate_response(summary_prompt, max_new_tokens=200)
print(f"Model summary:")
print(summary)
print()

# YOUR TASK: Compare the summary to the original text above.
# Mark each claim in the summary:
#   [F] = Faithful (directly supported by the text)
#   [H] = Hallucination (not in the original text)
#   [O] = Important point omitted from summary
#   [D] = Distortion (emphasis or framing shifted)
#
# Write your annotations below:
#
#
#


In [ ]:
#@title Solution 1b: Common summarization failure modes

print("Common failure modes to look for:")
print()
print("HALLUCINATION examples:")
print("  - The summary mentions specific numbers or dates not in the original")
print("  - The summary attributes positions to the party that are not stated")
print("  - The summary adds causal claims ('because of X, Labour proposed Y')")
print("    when the original text does not make that connection")
print()
print("OMISSION examples:")
print("  - The summary focuses on economic policy but drops healthcare/climate")
print("  - Specific funding amounts or targets are dropped")
print("  - Distinctive proposals (public ownership, specific institutions) are lost")
print()
print("DISTORTION examples:")
print("  - The summary makes a moderate position sound more extreme")
print("  - The summary frames proposals as 'promises' when the text says 'will consider'")
print("  - The relative emphasis between topics shifts")
print()
print("-> For research purposes, hallucination is the most dangerous failure mode")
print("   because it adds false information that looks authoritative. Always verify")
print("   extracted/summarized claims against the source text.")
print("   RAG (Section 2) helps by keeping the source text attached to the output.")


**Section takeaway:** Extraction and summarization are powerful but require verification. Unlike classification (where the output is constrained to a label set), generation can fail in ways that are difficult to detect without human review. For research, always evaluate faithfulness: does the output accurately reflect what the source text says?

---

*Take a 15-minute break. When we come back: how to work with corpora that are too large for any context window.*

---


# Section 2: Retrieval-Augmented Generation (RAG)

## The problem

Your research corpus has 500 pages of manifesto text across three parties and three elections. No model can process all of that at once: even models with 128k-token context windows would struggle with the full collection, and performance degrades with very long contexts.

## The solution: RAG

Retrieval-Augmented Generation works in four steps:

1. **Chunk:** Split your corpus into small, self-contained pieces (paragraphs, sections)
2. **Embed:** Convert each chunk into a vector using an embedding model (callback to Day 1: this is the same idea as word embeddings, applied to full passages)
3. **Index:** Store the vectors in a searchable index for fast retrieval
4. **Retrieve + Generate:** When a user asks a question, embed the question, find the most similar chunks, and pass them to the language model as context

The model never sees the full corpus. It only sees the most relevant pieces, selected by semantic similarity. This keeps the context small and the answers grounded in your actual data.


### Step 1: Chunking

Our corpus is already chunked (each manifesto excerpt is a paragraph). In practice, you would need to decide how to split your documents. Chunk size is a design choice:

- **Too small** (individual sentences): loses context, retrieval may return fragments
- **Too large** (full pages): dilutes the relevant signal with irrelevant text, uses more tokens
- **Right size** (paragraphs, ~100-300 words): self-contained units of meaning

For manifesto text, paragraph-level chunking works well. For other corpora (legal documents, parliamentary debates), you might chunk by section, speech turn, or a sliding window.


In [ ]:
# Our corpus is already paragraph-level chunks
print(f"Corpus: {len(corpus_df)} chunks")
print(f"Average chunk length: {corpus_df['text'].str.split().str.len().mean():.0f} words")
print(f"Range: {corpus_df['text'].str.split().str.len().min()}-{corpus_df['text'].str.split().str.len().max()} words")


### Step 2: Embed

We use a **sentence embedding model** to convert each chunk into a dense vector. Unlike the word embeddings from Day 1 (one vector per word), sentence embeddings capture the meaning of an entire passage in a single vector.

We use `all-MiniLM-L6-v2` from the `sentence-transformers` library: small (90MB), fast, and effective for retrieval tasks. It runs on CPU, so it does not compete with our language model for GPU memory.


In [ ]:
#@title Embed all chunks

from sentence_transformers import SentenceTransformer

# Load embedding model (runs on CPU, ~90MB)
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Embed all chunks
chunk_texts = corpus_df['text'].tolist()
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"Embedded {len(chunk_embeddings)} chunks")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")
print(f"Each chunk is now a vector of {chunk_embeddings.shape[1]} numbers")


### Step 3: Index with FAISS

[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search) is a library for fast nearest-neighbor search over dense vectors. For 28 chunks, numpy cosine similarity would work fine. But FAISS scales to millions of vectors, which is what you need for real research corpora.

The index stores all chunk embeddings and lets you quickly find the most similar ones to any query vector.


In [ ]:
#@title Build FAISS index

import faiss

# Normalize embeddings for cosine similarity
# (FAISS IndexFlatIP with normalized vectors = cosine similarity)
faiss.normalize_L2(chunk_embeddings)

# Build the index
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product (= cosine after normalization)
index.add(chunk_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dimension {embedding_dim}")
print(f"Index type: Flat (exact search). For larger corpora, use IndexIVFFlat or HNSW.")


### Exercise 2a: Retrieval

Now the core operation: given a research question, find the most relevant chunks. You embed the question with the same model, search the index, and retrieve the top-k results.

Your task: write research questions and evaluate whether the retriever finds the right passages.


In [ ]:
#@title Retrieval function

def retrieve(query, k=3):
    """
    Retrieve the top-k most relevant chunks for a given query.

    Args:
        query: A natural language question or search string.
        k: Number of chunks to retrieve.

    Returns:
        List of dicts with 'text', 'party', 'year', 'topic', and 'score'.
    """
    # Embed the query
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)

    # Search the index
    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = corpus_df.iloc[idx]
        results.append({
            'text': row['text'],
            'party': row['party'],
            'year': row['year'],
            'topic': row['topic'],
            'score': float(score),
            'id': row['id'],
        })
    return results


In [ ]:
### Exercise 2a: Test retrieval with research questions

# Try different research questions and evaluate the results.
# Does the retriever find the right passages?

queries = [
    "What is Labour's position on immigration?",
    "Which parties support increasing defence spending?",
    "What do the manifestos say about housing?",
    # Add your own queries:
    # "",
    # "",
]

for query in queries:
    if not query:
        continue
    print(f"QUERY: {query}")
    results = retrieve(query, k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] ({r['party']} {r['year']}, {r['topic']}) score={r['score']:.3f}")
        print(f"      {r['text'][:100]}...")
    print()


In [ ]:
#@title Solution 2a: Evaluating retrieval quality

print("Evaluating retrieval: things to look for")
print()
print("GOOD retrieval:")
print("  - 'Labour's position on immigration' returns Labour immigration excerpts")
print("  - 'Defence spending' returns defence/foreign policy excerpts")
print("  - Scores for relevant results are notably higher than irrelevant ones")
print()
print("PROBLEMATIC retrieval:")
print("  - A query about immigration returns an economy excerpt (topic mismatch)")
print("  - A query about Labour returns Conservative text (party mismatch)")
print("  - All results have similar scores (the embedding model cannot distinguish)")
print()
print("DESIGN CHOICES that affect retrieval quality:")
print("  - k (number of results): too few misses relevant info, too many adds noise")
print("  - Chunk size: if chunks are too broad, retrieval returns relevant + irrelevant text")
print("  - Query phrasing: 'immigration policy' vs 'border control' vs 'asylum seekers'")
print("    may retrieve different chunks. Try synonyms and see what changes.")


### Exercise 2b: The full RAG pipeline

Now combine retrieval with generation. The pipeline:
1. Embed the question
2. Retrieve the top-k chunks
3. Format them into a prompt alongside the question
4. Generate an answer grounded in the retrieved text

Your task: ask a research question, compare the RAG answer to the model's answer *without* retrieval, and evaluate which is more faithful.


In [ ]:
#@title RAG generation function

def rag_answer(query, k=3):
    """
    Answer a question using the RAG pipeline:
    retrieve relevant chunks, then generate an answer grounded in them.
    """
    # Step 1: Retrieve
    results = retrieve(query, k=k)

    # Step 2: Format context
    context_parts = []
    for r in results:
        context_parts.append(f"[{r['party']} {r['year']}, {r['topic']}]: {r['text']}")
    context = "\n\n".join(context_parts)

    # Step 3: Generate with context
    prompt = (
        "You are a research assistant analyzing UK party manifestos. "
        "Answer the question below using ONLY the provided manifesto excerpts. "
        "Cite which party and year you are drawing from. "
        "If the excerpts do not contain enough information to answer, say so.\n\n"
        f"MANIFESTO EXCERPTS:\n{context}\n\n"
        f"QUESTION: {query}\n\n"
        "ANSWER:"
    )

    answer = generate_response(prompt, max_new_tokens=300)
    return answer, results


In [ ]:
### Exercise 2b: RAG vs. no-RAG comparison

query = "What is Labour's position on public ownership?"  # change this

# RAG answer (grounded in retrieved passages)
print("=" * 60)
print(f"QUERY: {query}")
print("=" * 60)

rag_response, retrieved = rag_answer(query, k=3)
print(f"\nRAG ANSWER (grounded in {len(retrieved)} retrieved passages):")
print(rag_response)
print()
print("Retrieved passages:")
for r in retrieved:
    print(f"  [{r['party']} {r['year']}] {r['text'][:80]}...")
print()

# No-RAG answer (model's own knowledge)
no_rag_prompt = (
    "You are a research assistant analyzing UK party manifestos. "
    f"Answer this question: {query}"
)
no_rag_response = generate_response(no_rag_prompt, max_new_tokens=300)
print(f"NO-RAG ANSWER (model's own knowledge):")
print(no_rag_response)

print()
print("COMPARE: Which answer is more faithful to what the manifestos actually say?")
print("Does the no-RAG answer include claims not in any manifesto?")
print("Does the RAG answer stay grounded in the retrieved text?")


### Exercise 2c: Cross-party comparison

This is a realistic research application: use RAG to systematically compare party positions on the same issue across parties and elections.


In [ ]:
### Exercise 2c: Cross-party comparison via RAG

topic = "climate change and energy"  # change this to explore different topics

parties = ["Labour", "Conservative", "Liberal Democrats"]

print(f"CROSS-PARTY COMPARISON: {topic}")
print("=" * 60)

for party in parties:
    query = f"What is {party}'s position on {topic}?"
    answer, retrieved = rag_answer(query, k=2)

    # Check that retrieved chunks are actually from this party
    party_chunks = [r for r in retrieved if r['party'] == party]
    other_chunks = [r for r in retrieved if r['party'] != party]

    print(f"\n{party.upper()}")
    print(f"  Retrieved: {len(party_chunks)} from {party}, {len(other_chunks)} from other parties")
    print(f"  Answer: {answer[:250]}")
    if other_chunks:
        print(f"  [Note: retriever also pulled chunks from: {[r['party'] for r in other_chunks]}]")
    print()

print("REFLECTION:")
print("  - Did the retriever correctly match each party to its own manifesto text?")
print("  - Are there cases where party A's text was retrieved for party B's query?")
print("  - How would you improve this? (Hint: metadata filtering before retrieval)")


In [ ]:
#@title Stretch: Metadata-filtered retrieval

def retrieve_filtered(query, k=3, party=None, year=None, topic=None):
    """Retrieve with optional metadata filters applied before semantic search."""
    # Filter the corpus first
    mask = pd.Series([True] * len(corpus_df))
    if party:
        mask &= corpus_df['party'] == party
    if year:
        mask &= corpus_df['year'] == year
    if topic:
        mask &= corpus_df['topic'] == topic

    filtered_indices = corpus_df[mask].index.tolist()
    if not filtered_indices:
        return []

    # Get embeddings for filtered chunks only
    filtered_embeddings = chunk_embeddings[filtered_indices]

    # Build a temporary index
    temp_index = faiss.IndexFlatIP(embedding_dim)
    temp_index.add(filtered_embeddings)

    # Search
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, local_indices = temp_index.search(query_vec, min(k, len(filtered_indices)))

    results = []
    for score, local_idx in zip(scores[0], local_indices[0]):
        global_idx = filtered_indices[local_idx]
        row = corpus_df.iloc[global_idx]
        results.append({
            'text': row['text'], 'party': row['party'],
            'year': row['year'], 'topic': row['topic'],
            'score': float(score), 'id': row['id'],
        })
    return results


# Example: compare Labour's position on the economy across elections
print("Labour's economic position over time:")
print("=" * 60)
for year in [2017, 2019, 2024]:
    results = retrieve_filtered("economic policy and spending", k=1, party="Labour", year=year)
    if results:
        r = results[0]
        print(f"\n{year}: {r['text'][:200]}")
    else:
        print(f"\n{year}: No matching excerpts found")


**Section takeaway:** RAG transforms the model from a (unreliable) knowledge source into a retriever + synthesizer grounded in your actual data. The pipeline has four design choices that affect quality: chunk size, embedding model, number of retrieved passages (k), and whether to apply metadata filters. For research, always verify that retrieved passages are relevant and that the generated answer is faithful to them.

---


# Section 3: LLMs as Simulated Agents

## The homo silicus paradigm

Can language models stand in for human survey respondents? The idea is seductive: if you can prompt an LLM to "be" a 55-year-old conservative woman from the Midlands, you could generate survey responses for any demographic at any scale, instantly and cheaply. No recruitment, no attrition, no cost per respondent.

This is called **silicon sampling** (Argyle et al., 2023): using LLMs to simulate human populations. The promise is real, and so are the problems.

**What works:**
- LLMs can reproduce aggregate opinion distributions for well-studied topics (e.g., party identification predicting policy preferences in the US)
- Persona prompting shifts responses in the expected direction (telling the model "you are a Democrat" produces more liberal responses)
- For pilot testing survey instruments, simulated responses can identify confusing questions or unexpected patterns

**What fails:**
- **WEIRD bias:** LLMs are trained primarily on English-language, Western, educated, internet-connected text. Simulating respondents outside this distribution is unreliable.
- **Sycophancy:** models tend to agree with the framing of the question, inflating agreement rates
- **Anchoring:** the order and framing of response options affects simulated responses more than real ones
- **Flat distributions:** for niche or controversial topics, LLMs often produce uncommitted, centrist responses rather than the polarized distributions observed in real data
- **Refusal:** safety training causes models to refuse questions about sensitive topics that real respondents would answer


### [PLACEHOLDER: Simulation exercises]

This section will use survey data and tools from an ongoing research project on synthetic survey responses. The exercises will have students:

1. Explore real survey questions with known demographic breakdowns
2. Construct persona prompts from respondent metadata
3. Generate simulated responses and compare to real distributions
4. Test where simulation breaks down (edge cases, niche topics, sensitive questions)

*Details to be filled in once the simulation data and tooling are finalized.*


---

# Closing

## What you learned today

1. **Information extraction** turns unstructured text into structured data, but requires faithfulness evaluation. The model can hallucinate, omit, or distort, and these failures are harder to detect than classification errors.

2. **RAG** grounds the model's outputs in your actual data. The pipeline (chunk, embed, index, retrieve, generate) is the standard approach for working with corpora too large for any context window. Design choices at every step (chunk size, embedding model, k, metadata filters) affect quality.

3. **Simulated agents** can reproduce aggregate patterns for well-studied populations but fail on niche topics, non-Western demographics, and sensitive questions. They are tools for pilot testing and hypothesis generation, not replacements for real human data.

| Day | Theme | What You Learn |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | How models represent meaning, process text, and generate language |
| Day 2 ✓ | From Models to Tools | Post-training (RLHF, DPO), prompting as experimental design, model evaluation |
| Day 3 ✓ | Deploying for Research | Validation, fine-tuning (BERT and LoRA), APIs and working at scale |
| Day 4 ✓ | Social Science Applications | Information extraction, RAG, LLMs as simulated agents |
| **→ Day 5** | **Agentic Workflows** | **Tool use, autonomous research assistants, capstone project** |

## Bridge to Day 5

Over four days you have built a toolkit: embeddings, prompting, validation, fine-tuning, extraction, RAG. Tomorrow you learn how to **connect these pieces into autonomous workflows**. The RAG pipeline becomes a tool an agent can call. The extraction prompts become functions in a planning loop. The model stops being a tool you operate and starts being an assistant that operates tools on your behalf.

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)
